In [3]:
import pandas as pd

df_raw = pd.read_excel('dataset_circor.xlsx')  # ajusta la ruta si esta en otro lado
ciclos_por_archivo = df_raw.groupby('archivo').size()

for umbral in [2, 3, 5, 8]:
    archivos_ok = ciclos_por_archivo[ciclos_por_archivo >= umbral]
    print(f"umbral >= {umbral} ciclos: quedan {len(archivos_ok)} de {len(ciclos_por_archivo)} archivos ({100*len(archivos_ok)/len(ciclos_por_archivo):.0f}%)")

umbral >= 2 ciclos: quedan 1079 de 1079 archivos (100%)
umbral >= 3 ciclos: quedan 1079 de 1079 archivos (100%)
umbral >= 5 ciclos: quedan 1079 de 1079 archivos (100%)
umbral >= 8 ciclos: quedan 649 de 1079 archivos (60%)


In [4]:
for umbral in [2, 3, 5, 8]:
    archivos_ok = ciclos_por_archivo[ciclos_por_archivo >= umbral].index
    etiquetas_ok = df_raw[df_raw['archivo'].isin(archivos_ok)].drop_duplicates('archivo')['Etiqueta']
    print(f"umbral >= {umbral}: Sano={sum(etiquetas_ok==0)}  Soplo={sum(etiquetas_ok==2)}")

umbral >= 2: Sano=826  Soplo=253
umbral >= 3: Sano=826  Soplo=253
umbral >= 5: Sano=826  Soplo=253
umbral >= 8: Sano=508  Soplo=141


In [5]:
df_circor = pd.read_excel('dataset_circor.xlsx')  # ajusta la ruta

feature_cols_circor = [c for c in df_circor.columns if c not in ('Etiqueta', 'archivo', 'paciente_id')]
X_circor = df_circor[feature_cols_circor].values
y_circor = df_circor['Etiqueta'].values
grupos_circor = df_circor['paciente_id'].values

print(f"{df_circor.shape[0]} filas de {df_circor['paciente_id'].nunique()} pacientes")
print("\npacientes por clase:")
print(df_circor.groupby('Etiqueta')['paciente_id'].nunique())

11153 filas de 586 pacientes

pacientes por clase:
Etiqueta
0    459
2    127
Name: paciente_id, dtype: int64


In [1]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score, accuracy_score
import joblib

df_v2 = pd.read_excel('dataset_circor_v2.xlsx')
ciclos_por_archivo = df_v2.groupby('archivo').size()
print(ciclos_por_archivo.describe())
print(f"\narchivos con menos de 5 ciclos: {(ciclos_por_archivo < 5).sum()} de {len(ciclos_por_archivo)}")

count    3005.000000
mean       29.715141
std        11.681073
min         1.000000
25%        22.000000
50%        28.000000
75%        37.000000
max        87.000000
dtype: float64

archivos con menos de 5 ciclos: 23 de 3005


In [4]:
df_v2 = pd.read_excel('dataset_circor_v2.xlsx')

ciclos_por_archivo = df_v2.groupby('archivo').size()
archivos_ok = ciclos_por_archivo[ciclos_por_archivo >= 5].index
df_v2 = df_v2[df_v2['archivo'].isin(archivos_ok)]
print(f"tras filtro de calidad: {df_v2.shape[0]} filas, {df_v2['paciente_id'].nunique()} pacientes")

feature_cols_v2 = [c for c in df_v2.columns if c not in ('Etiqueta', 'archivo', 'paciente_id')]
df_v2['paciente_id'] = df_v2['paciente_id'].astype(str)

pacientes_prueba = pd.read_csv('pacientes_prueba_final.csv')['paciente_id'].astype(str)
es_prueba = df_v2['paciente_id'].isin(pacientes_prueba)

df_dev_v2 = df_v2[~es_prueba]
df_prueba_v2 = df_v2[es_prueba]

print(f"dev:           {df_dev_v2.shape[0]} filas, {df_dev_v2['paciente_id'].nunique()} pacientes")
print(f"prueba final:  {df_prueba_v2.shape[0]} filas, {df_prueba_v2['paciente_id'].nunique()} pacientes")

tras filtro de calidad: 89238 filas, 873 pacientes
dev:           76630 filas, 756 pacientes
prueba final:  12608 filas, 117 pacientes


In [6]:
X_dev_v2 = df_dev_v2[feature_cols_v2].values
y_dev_v2 = df_dev_v2['Etiqueta'].values
grupos_dev_v2 = df_dev_v2['paciente_id'].values

sgkf_v2 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)

modelos = {
    'Regresion Logistica': LogisticRegression(max_iter=2000, class_weight='balanced'),
    'SVM': SVC(kernel='rbf', class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=0),
    'Gradient Boosting': GradientBoostingClassifier(random_state=0),
}

for nombre, modelo in modelos.items():
    pipe = Pipeline([('escalador', StandardScaler()), ('clf', modelo)])
    scores = cross_val_score(pipe, X_dev_v2, y_dev_v2, cv=sgkf_v2, groups=grupos_dev_v2, scoring='accuracy')
    print(f"{nombre:22s} exactitud = {scores.mean():.3f} +/- {scores.std():.3f}")

Regresion Logistica    exactitud = 0.695 +/- 0.016
SVM                    exactitud = 0.738 +/- 0.009
Random Forest          exactitud = 0.841 +/- 0.020
Gradient Boosting      exactitud = 0.843 +/- 0.022


In [6]:
X_dev_v2 = df_dev_v2[feature_cols_v2].values
y_dev_v2 = df_dev_v2['Etiqueta'].values
grupos_dev_v2 = df_dev_v2['paciente_id'].values

sgkf_v2 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)

mejor_pipe_v2 = Pipeline([('escalador', StandardScaler()), ('clf', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=0))])
proba_dev_v2 = cross_val_predict(mejor_pipe_v2, X_dev_v2, y_dev_v2, cv=sgkf_v2, groups=grupos_dev_v2, method='predict_proba')
print("listo")

listo


In [7]:
def voto_mayoria(grupo):
    return grupo.value_counts().idxmax()

df_dev_v2 = df_dev_v2.copy()
df_dev_v2['prob_soplo'] = proba_dev_v2[:, 1]
etiquetas_dev_paciente_v2 = df_dev_v2.groupby('paciente_id')['Etiqueta'].first()

print(f"{'umbral':>8} {'sensibilidad':>13} {'especificidad':>15} {'Youden J':>10}")
mejor_j_v2, mejor_umbral_v2 = -1, None
for umbral in [0.5, 0.4, 0.3, 0.25, 0.2, 0.15, 0.1]:
    df_dev_v2['pred_ciclo'] = (df_dev_v2['prob_soplo'] >= umbral).astype(int) * 2
    pred_paciente = df_dev_v2.groupby('paciente_id')['pred_ciclo'].apply(voto_mayoria)
    m = confusion_matrix(etiquetas_dev_paciente_v2, pred_paciente)
    tn, fp, fn, vp = m.ravel()
    sens, esp = vp/(vp+fn), tn/(tn+fp)
    j = sens + esp - 1
    print(f"{umbral:>8.2f} {sens:>13.3f} {esp:>15.3f} {j:>10.3f}")
    if j > mejor_j_v2:
        mejor_j_v2, mejor_umbral_v2 = j, umbral

print(f"\numbral elegido (mejor Youden J): {mejor_umbral_v2}")

  umbral  sensibilidad   especificidad   Youden J
    0.50         0.159           0.998      0.157
    0.40         0.214           0.995      0.209
    0.30         0.297           0.982      0.279
    0.25         0.414           0.964      0.378
    0.20         0.552           0.903      0.455
    0.15         0.703           0.715      0.419
    0.10         0.917           0.355      0.272

umbral elegido (mejor Youden J): 0.2


In [9]:
import numpy as np

np.save('proba_dev_v2.npy', proba_dev_v2)
print("guardado")

guardado


In [10]:
X_prueba_v2 = df_prueba_v2[feature_cols_v2].values
y_prueba_v2 = df_prueba_v2['Etiqueta'].values
grupos_prueba_v2 = df_prueba_v2['paciente_id'].values

modelo_dev_v2 = Pipeline([('escalador', StandardScaler()), ('clf', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=0))])
modelo_dev_v2.fit(X_dev_v2, y_dev_v2)

proba_final_v2 = modelo_dev_v2.predict_proba(X_prueba_v2)[:, 1]

np.save('proba_final_v2.npy', proba_final_v2)
print("listo y guardado")

listo y guardado


In [11]:
df_prueba_v2 = df_prueba_v2.copy()
df_prueba_v2['prob_soplo'] = proba_final_v2
df_prueba_v2['pred_ciclo'] = (df_prueba_v2['prob_soplo'] >= mejor_umbral_v2).astype(int) * 2

pred_paciente_final_v2 = df_prueba_v2.groupby('paciente_id')['pred_ciclo'].apply(voto_mayoria)
etiquetas_paciente_final_v2 = df_prueba_v2.groupby('paciente_id')['Etiqueta'].first()

m_final_v2 = confusion_matrix(etiquetas_paciente_final_v2, pred_paciente_final_v2)
tn, fp, fn, vp = m_final_v2.ravel()
print(f"EXAMEN FINAL v2, con {len(etiquetas_paciente_final_v2)} pacientes que el modelo nunca vio:")
print(f"Sensibilidad: {vp/(vp+fn):.3f}   Especificidad: {tn/(tn+fp):.3f}")
print(m_final_v2)

EXAMEN FINAL v2, con 117 pacientes que el modelo nunca vio:
Sensibilidad: 0.588   Especificidad: 0.892
[[74  9]
 [14 20]]


In [13]:
auc_test_v2 = roc_auc_score(y_prueba_v2, proba_final_v2)
print(f"ROC-AUC en la prueba final (por ciclo): {auc_test_v2:.3f}")

pred_ciclo_05 = (proba_final_v2 >= 0.5).astype(int) * 2
acc_ciclo_test = accuracy_score(y_prueba_v2, pred_ciclo_05)
print(f"Exactitud por ciclo en prueba final (umbral 0.5): {acc_ciclo_test:.3f}  (en dev fue 0.841)")

print(f"\n{'umbral':>8} {'sensibilidad':>13} {'especificidad':>15} {'Youden J':>10}")
for umbral in [0.5, 0.4, 0.3, 0.25, 0.2, 0.15, 0.1]:
    pred_c = (df_prueba_v2['prob_soplo'] >= umbral).astype(int) * 2
    df_prueba_v2['pred_temp'] = pred_c
    pred_p = df_prueba_v2.groupby('paciente_id')['pred_temp'].apply(voto_mayoria)
    m = confusion_matrix(etiquetas_paciente_final_v2, pred_p)
    tn, fp, fn, vp = m.ravel()
    sens, esp = vp/(vp+fn), tn/(tn+fp)
    print(f"{umbral:>8.2f} {sens:>13.3f} {esp:>15.3f} {sens+esp-1:>10.3f}")

ROC-AUC en la prueba final (por ciclo): 0.698
Exactitud por ciclo en prueba final (umbral 0.5): 0.741  (en dev fue 0.841)

  umbral  sensibilidad   especificidad   Youden J
    0.50         0.147           1.000      0.147
    0.40         0.235           1.000      0.235
    0.30         0.412           1.000      0.412
    0.25         0.500           0.976      0.476
    0.20         0.588           0.892      0.480
    0.15         0.676           0.699      0.375
    0.10         0.912           0.301      0.213
